# Common Libraries

In [1]:
import os, mujoco
import numpy as np
import ipywidgets as widgets
from mink import Configuration, SE3, SO3, FrameTask, solve_ik
from IPython.display import display, clear_output
from scipy.spatial.transform import Rotation as R
import mediapy as media

# Model specification

In [2]:
dir_path = "/Users/seojin/mink/examples"
model_path = os.path.join(dir_path, "franka_emika_panda/mjx_scene.xml")
model = mujoco.MjModel.from_xml_path(model_path)

# Simulation

In [3]:
configuration = Configuration(model)
configuration.update_from_keyframe("home")

renderer = mujoco.Renderer(configuration.model, height=400, width=600)

duration = 2.0  # seconds
fps = 60
n_frames = int(duration * fps)
gain = 1.0 - 0.01 ** (1.0 / n_frames)  # ≈ 0.038

task = FrameTask(
    frame_name = "attachment_site",
    frame_type = "site",
    position_cost = 1.0,
    orientation_cost = 1.0,
    gain = gain,
)

In [4]:
def update_robot(x, y, z,
                 w_o, x_o, y_o, z_o):
    # Make new target
    new_translation = np.array([x, y, z])
    orientation = np.array([w_o, x_o, y_o, z_o])
    target_val = np.concatenate([orientation, new_translation])
    new_target = SE3(wxyz_xyz = target_val)

    # Set target position and orientation
    task.set_target(new_target)
    configuration.data.mocap_pos[0] = new_target.wxyz_xyz[4:]
    configuration.data.mocap_quat[0] = new_target.wxyz_xyz[:4]
    
    # Rendering
    formatted_text = ", ".join([f"{val:.2f}" for val in target_val])
    data_label.value = f"wxyz_xyz: [{formatted_text}]"

    mujoco.mj_forward(configuration.model, configuration.data)
    renderer.update_scene(configuration.data)
    pixels = renderer.render()
    
    with output_area:
        clear_output(wait=True)
        media.show_image(pixels)

def run_simulation(b):
    frames = []
    dt = 1.0 / fps
    solver_name = "quadprog" 

    for _ in range(n_frames):
        # Perform inverse kinematics
        vel = solve_ik(configuration, [task], dt, solver=solver_name)
        configuration.integrate_inplace(vel, dt)
        
        # Update the physical state of the simulation
        mujoco.mj_forward(configuration.model, configuration.data)
        
        # Save the current frame for video
        renderer.update_scene(configuration.data)
        pixels = renderer.render()
        frames.append(pixels)

    with output_area:
        clear_output(wait=True)
        media.show_video(frames, fps=fps, loop = False)
    
# Make widgets
data_label = widgets.Label(value="wxyz_xyz: [0, 0, 0, 0, 0, 0, 0]")
x_slider = widgets.FloatSlider(value=0.2, min=-1.0, max=1.0, step=0.01, description='Target X:')
y_slider = widgets.FloatSlider(value=0.3, min=-1.0, max=1.0, step=0.01, description='Target Y:')
z_slider = widgets.FloatSlider(value=0.3, min=-1.0, max=1.0, step=0.01, description='Target Z:')

w_o_slider = widgets.FloatSlider(value=0, min=-1.0, max=1.0, step=0.01, description='Target w_o:')
x_o_slider = widgets.FloatSlider(value=0, min=-1.0, max=1.0, step=0.01, description='Target x_o:')
y_o_slider = widgets.FloatSlider(value=0, min=-1.0, max=1.0, step=0.01, description='Target y_o:')
z_o_slider = widgets.FloatSlider(value=0, min=-1.0, max=1.0, step=0.01, description='Target z_o:')

sim_button = widgets.Button(
    description='Run Simulation',
    button_style='success', # 초록색 버튼
    icon='play'
)

output_area = widgets.Output()

# Layout and event connection
ui = widgets.VBox([
    widgets.HBox([data_label, w_o_slider]),
    widgets.HBox([x_slider, x_o_slider]),
    widgets.HBox([y_slider, y_o_slider]),
    widgets.HBox([z_slider, z_o_slider]),
    sim_button,
])

out = widgets.interactive_output(update_robot, {
    'x': x_slider, 'y': y_slider, 'z': z_slider,
    'w_o' : w_o_slider, 'x_o': x_o_slider, 'y_o': y_o_slider, 'z_o': z_o_slider,
})

sim_button.on_click(run_simulation)

display(ui, output_area)
update_robot(x_slider.value, y_slider.value, z_slider.value, 
             w_o_slider.value, x_o_slider.value, y_o_slider.value, z_o_slider.value)

Output()